# Probability Fundamentals

## Learning Objectives
1. Compute joint, marginal, and conditional probabilities from first principles
2. Verify Bayes' theorem numerically and understand base rate neglect
3. Build a 2D joint distribution and test independence with KL divergence
4. Apply probability reasoning to spam filters, A/B tests, and medical diagnosis

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from collections import defaultdict

# Reproducibility
np.random.seed(42)

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Libraries loaded successfully')
print(f'NumPy version: {np.__version__}')
print('Ready to compute probabilities from scratch')

## Level 1: Core Probability Operations

We simulate a standard 52-card deck to compute exact probabilities via counting,
then verify Bayes' theorem numerically.

Key formulas:
- P(A) = |A| / |Omega|
- P(A union B) = P(A) + P(B) - P(A intersection B)
- P(A | B) = P(A intersection B) / P(B)
- Bayes: P(A|B) = P(B|A) * P(A) / P(B)

In [ ]:
# --- Card deck simulation ---
# Each card: (rank, suit)
ranks = list(range(1, 14))   # 1=Ace, 11=J, 12=Q, 13=K
suits = ['hearts', 'diamonds', 'clubs', 'spades']
deck = [(r, s) for s in suits for r in ranks]  # 52 cards
N = len(deck)
print(f'Deck size: {N} cards')

# Event A: card is a Heart
A = [(r, s) for (r, s) in deck if s == 'hearts']
# Event B: card is a Face card (J/Q/K)
B = [(r, s) for (r, s) in deck if r >= 11]
# Intersection A and B
A_and_B = [(r, s) for (r, s) in deck if s == 'hearts' and r >= 11]
# Union A or B
A_or_B = list(set(A) | set(B))

P_A = len(A) / N
P_B = len(B) / N
P_AandB = len(A_and_B) / N
P_AorB = len(A_or_B) / N

print(f'P(Heart)          = {P_A:.4f}  (exact: 13/52 = {13/52:.4f})')
print(f'P(Face card)      = {P_B:.4f}  (exact: 12/52 = {12/52:.4f})')
print(f'P(Heart AND Face) = {P_AandB:.4f}  (exact:  3/52 = {3/52:.4f})')
print(f'P(Heart OR  Face) = {P_AorB:.4f}  (exact: 22/52 = {22/52:.4f})')

# Verify addition rule: P(A or B) = P(A) + P(B) - P(A and B)
computed = P_A + P_B - P_AandB
print(f'Addition rule check: P(A)+P(B)-P(A^B) = {computed:.4f}  matches: {abs(computed - P_AorB) < 1e-10}')

# --- Conditional probability ---
P_A_given_B = P_AandB / P_B   # Hearts given Face card
P_B_given_A = P_AandB / P_A   # Face card given Heart

print(f'P(Heart | Face)  = {P_A_given_B:.4f}  (3 face hearts / 12 face cards = {3/12:.4f})')
print(f'P(Face | Heart)  = {P_B_given_A:.4f}  (3 face hearts / 13 hearts = {3/13:.4f})')
print('NOTE: P(A|B) != P(B|A) -- this asymmetry causes base rate neglect')

# --- Verify Bayes theorem ---
# P(A|B) = P(B|A) * P(A) / P(B)
bayes_P_A_given_B = (P_B_given_A * P_A) / P_B
print(f'Bayes check: P(Heart|Face) via Bayes = {bayes_P_A_given_B:.4f}  direct = {P_A_given_B:.4f}')
print(f'Match: {abs(bayes_P_A_given_B - P_A_given_B) < 1e-10}')

# --- Monte Carlo verification ---
n_trials = 100_000
draw_indices = np.random.randint(0, N, n_trials)
draws = [deck[i] for i in draw_indices]
mc_P_A = sum(1 for (r, s) in draws if s == 'hearts') / n_trials
mc_P_B = sum(1 for (r, s) in draws if r >= 11) / n_trials
mc_P_AandB = sum(1 for (r, s) in draws if s == 'hearts' and r >= 11) / n_trials
mc_P_A_given_B = mc_P_AandB / mc_P_B

print(f'Monte Carlo ({n_trials:,} trials):')
print(f'  P(Heart)      MC={mc_P_A:.4f}  exact={P_A:.4f}  error={abs(mc_P_A-P_A):.4f}')
print(f'  P(Face)       MC={mc_P_B:.4f}  exact={P_B:.4f}  error={abs(mc_P_B-P_B):.4f}')
print(f'  P(Heart|Face) MC={mc_P_A_given_B:.4f}  exact={P_A_given_B:.4f}  error={abs(mc_P_A_given_B-P_A_given_B):.4f}')

# --- Multiplication rule check ---
# P(A and B) = P(A|B) * P(B) = P(B|A) * P(A)
mult1 = P_A_given_B * P_B
mult2 = P_B_given_A * P_A
print(f'Multiplication rule: P(A|B)*P(B)={mult1:.4f}  P(B|A)*P(A)={mult2:.4f}  direct={P_AandB:.4f}')
print(f'Both equal P(A and B): {abs(mult1 - P_AandB) < 1e-10}')

## Level 2: Joint, Marginal, and Conditional Distributions

We construct a 2D joint distribution over discrete random variables,
then compute marginals, conditionals, and test for independence using KL divergence.

The **independence test**: if P(A,B) = P(A)*P(B) everywhere, then KL(P(A,B) || P(A)P(B)) = 0.

In [ ]:
# --- 2D joint distribution ---
# Variable X: weather = {sunny, cloudy, rainy}  (3 states, rows)
# Variable Y: commute = {short, medium, long}   (3 states, cols)
joint = np.array([
    [0.20, 0.10, 0.05],   # sunny
    [0.10, 0.15, 0.10],   # cloudy
    [0.05, 0.08, 0.17],   # rainy
])
weather_labels = ['Sunny', 'Cloudy', 'Rainy']
commute_labels = ['Short', 'Medium', 'Long']

print(f'Joint table sums to: {joint.sum():.6f} (should be 1.0)')

# --- Marginal distributions ---
# P(Weather) = sum over commute (axis=1 for rows)
P_weather = joint.sum(axis=1)
# P(Commute) = sum over weather (axis=0 for columns)
P_commute = joint.sum(axis=0)

print('Marginal P(Weather):')
for label, p in zip(weather_labels, P_weather):
    print(f'  P({label}) = {p:.4f}')

print('Marginal P(Commute):')
for label, p in zip(commute_labels, P_commute):
    print(f'  P({label}) = {p:.4f}')

# --- Conditional: P(Commute | Weather=Rainy) ---
# P(Y | X=rainy) = P(X=rainy, Y) / P(X=rainy) = row 2 / P_weather[2]
P_commute_given_rainy = joint[2, :] / P_weather[2]
print('P(Commute | Weather=Rainy):')
for label, p in zip(commute_labels, P_commute_given_rainy):
    print(f'  P(Commute={label} | Rainy) = {p:.4f}')

# --- Independence check ---
# If X and Y independent: P(X,Y) = P(X) * P(Y) everywhere
joint_if_indep = np.outer(P_weather, P_commute)  # outer product
max_dev = np.max(np.abs(joint - joint_if_indep))
print(f'Max deviation from independence: {max_dev:.4f} (non-zero => dependent)')

# --- KL divergence: KL(joint || product_of_marginals) ---
# Zero iff independent; positive means variables share information
def kl_divergence(P_flat, Q_flat):
    # Stable KL: only compute where P > 0
    mask = P_flat > 0
    return float(np.sum(P_flat[mask] * np.log(P_flat[mask] / Q_flat[mask])))

kl = kl_divergence(joint.flatten(), joint_if_indep.flatten())
print(f'KL(joint || product of marginals) = {kl:.6f}')
print(f'  KL > 0 confirms weather and commute are NOT independent')

# --- Mutual information = KL of joint from product-of-marginals ---
# MI = E[log P(X,Y) / (P(X)P(Y))]  -- same as the KL above
print(f'Mutual information I(Weather; Commute) = {kl:.6f} nats')

# --- Plot heatmaps ---
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, data, title, cmap in zip(
    axes,
    [joint, joint_if_indep, joint - joint_if_indep],
    ['Joint P(W,C)', 'Product P(W)*P(C)', 'Difference'],
    ['Blues', 'Greens', 'RdBu'],
):
    vmax = 0.1 if 'Diff' in title else None
    im = ax.imshow(data, cmap=cmap, aspect='auto', vmin=-vmax if vmax else None, vmax=vmax)
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(range(3)); ax.set_xticklabels(commute_labels, fontsize=8)
    ax.set_yticks(range(3)); ax.set_yticklabels(weather_labels, fontsize=8)
    for i in range(3):
        for j in range(3):
            val = data[i, j]
            ax.text(j, i, f'{val:+.2f}' if 'Diff' in title else f'{val:.2f}',
                    ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=ax)

plt.suptitle('Joint Distribution Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('stats_01_joint.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_01_joint.png')

## Real-World Example 1: Spam Filter with Naive Bayes

Classic application: compute P(spam | email words) using Bayes' theorem.
- **Prior**: P(spam) from training set base rate
- **Likelihood**: P(word | spam) estimated from word counts with Laplace smoothing
- **Posterior**: P(spam | words) via Bayes, normalized to sum to 1

Key insight: work in log-space to avoid underflow when multiplying many small probabilities.

In [ ]:
# --- Training corpus ---
training_emails = [
    (['money', 'win', 'prize', 'free', 'click'], 'spam'),
    (['money', 'transfer', 'now', 'free'], 'spam'),
    (['win', 'lottery', 'free', 'prize'], 'spam'),
    (['click', 'here', 'money', 'now'], 'spam'),
    (['win', 'free', 'cash', 'money'], 'spam'),
    (['meeting', 'tomorrow', 'agenda', 'report'], 'ham'),
    (['report', 'deadline', 'review', 'meeting'], 'ham'),
    (['lunch', 'tomorrow', 'meeting', 'agenda'], 'ham'),
    (['project', 'update', 'report', 'review'], 'ham'),
    (['coffee', 'chat', 'tomorrow', 'lunch'], 'ham'),
]

n_spam  = sum(1 for _, lbl in training_emails if lbl == 'spam')
n_ham   = sum(1 for _, lbl in training_emails if lbl == 'ham')
n_total = len(training_emails)
P_spam_prior = n_spam / n_total
P_ham_prior  = n_ham  / n_total
print(f'Prior P(spam) = {P_spam_prior:.2f},  P(ham) = {P_ham_prior:.2f}')

# --- Word count tables ---
word_counts = {'spam': defaultdict(int), 'ham': defaultdict(int)}
word_totals = {'spam': 0, 'ham': 0}
for words, label in training_emails:
    for w in words:
        word_counts[label][w] += 1
        word_totals[label] += 1

# Vocabulary (all unique words)
vocabulary = set(w for words, _ in training_emails for w in words)
V = len(vocabulary)
print(f'Vocabulary size: {V} unique words')

# --- Laplace-smoothed P(word | class) ---
def word_likelihood(word, label, alpha=1.0):
    # Laplace smoothing: add alpha to count, alpha*V to denominator
    return (word_counts[label][word] + alpha) / (word_totals[label] + alpha * V)

# --- Naive Bayes classifier ---
def classify_email(words, verbose=True):
    # Work in log-space to avoid underflow
    log_post_spam = np.log(P_spam_prior) + sum(np.log(word_likelihood(w, 'spam')) for w in words)
    log_post_ham  = np.log(P_ham_prior)  + sum(np.log(word_likelihood(w, 'ham'))  for w in words)
    # Normalize via log-sum-exp
    log_normalizer = np.logaddexp(log_post_spam, log_post_ham)
    p_spam = float(np.exp(log_post_spam - log_normalizer))
    p_ham  = float(np.exp(log_post_ham  - log_normalizer))
    prediction = 'spam' if p_spam > 0.5 else 'ham'
    if verbose:
        print(f'  Words: {words}')
        print(f'  P(spam|email) = {p_spam:.4f},  P(ham|email) = {p_ham:.4f}')
        print(f'  Predicted: {prediction}')
    return prediction, p_spam

# --- Classify test emails ---
test_cases = [
    ['money', 'free', 'win'],
    ['meeting', 'tomorrow', 'lunch'],
    ['money', 'meeting'],        # ambiguous
    ['click', 'report', 'free'], # mixed signals
]
print('--- Email Classification Results ---')
for words in test_cases:
    print()
    classify_email(words)

# --- Show word likelihood ratios (spam signal strength) ---
print('--- Word Spam Signal Strength (log P(w|spam)/P(w|ham)) ---')
key_words = ['money', 'free', 'win', 'meeting', 'report', 'tomorrow']
for w in key_words:
    ratio = np.log(word_likelihood(w, 'spam') / word_likelihood(w, 'ham'))
    print(f'  {w:<12}  log ratio = {ratio:+.2f}  ({'spam signal' if ratio > 0 else 'ham signal'})')

## Real-World Example 2: A/B Test -- Frequentist p-value vs Bayesian P(B > A)

A p-value is NOT the probability that B is better than A. It is the probability
of seeing data this extreme if the null hypothesis were true.

Bayesian P(B > A | data) is the more actionable quantity for ship/no-ship decisions.
We compare both approaches on the same simulated experiment.

In [ ]:
np.random.seed(42)

# --- Simulated A/B experiment ---
true_rate_A = 0.10   # true conversion rate control
true_rate_B = 0.12   # true conversion rate treatment
n_per_group = 1000

conv_A = np.random.binomial(n=1, p=true_rate_A, size=n_per_group)
conv_B = np.random.binomial(n=1, p=true_rate_B, size=n_per_group)
obs_A = conv_A.mean()
obs_B = conv_B.mean()
print(f'Observed rate A: {obs_A:.4f} (true {true_rate_A})')
print(f'Observed rate B: {obs_B:.4f} (true {true_rate_B})')
print(f'Relative lift: {(obs_B - obs_A)/obs_A*100:.1f}%')

# --- Frequentist: two-proportion z-test ---
p_pool = (conv_A.sum() + conv_B.sum()) / (2 * n_per_group)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_per_group + 1/n_per_group))
z_stat = (obs_B - obs_A) / se
from scipy.stats import norm as norm_dist
p_value = 2 * (1 - norm_dist.cdf(abs(z_stat)))  # two-tailed
print(f'Frequentist z-test: z={z_stat:.3f}, p-value={p_value:.4f}')
print(f'Significant (alpha=0.05)? {p_value < 0.05}')
print('CAUTION: p-value != P(B is better than A)')

# --- Bayesian: Beta-Binomial posteriors ---
# Prior: Beta(1,1) = uniform (no prior knowledge)
alpha0, beta0 = 1, 1
n_A, k_A = n_per_group, int(conv_A.sum())
n_B, k_B = n_per_group, int(conv_B.sum())

# Posterior: Beta(alpha0+k, beta0+n-k)
post_A = stats.beta(alpha0 + k_A, beta0 + n_A - k_A)
post_B = stats.beta(alpha0 + k_B, beta0 + n_B - k_B)
print(f'Posterior A: Beta({alpha0+k_A}, {beta0+n_A-k_A})')
print(f'Posterior B: Beta({alpha0+k_B}, {beta0+n_B-k_B})')

# --- P(B > A) via Monte Carlo from posteriors ---
n_mc = 100_000
samp_A = post_A.rvs(n_mc, random_state=0)
samp_B = post_B.rvs(n_mc, random_state=1)
P_B_beats_A = float(np.mean(samp_B > samp_A))
lift_samples = samp_B - samp_A

print(f'P(B > A | data) = {P_B_beats_A:.4f}  ({n_mc:,} MC samples)')
print(f'Expected lift = {lift_samples.mean()*100:.2f}%')
print(f'95% CI on lift: [{np.percentile(lift_samples,2.5)*100:.2f}%, {np.percentile(lift_samples,97.5)*100:.2f}%]')

# --- Plot: posterior distributions and lift distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.linspace(0.05, 0.19, 400)
axes[0].plot(x, post_A.pdf(x), 'b-', lw=2, label=f'Posterior A (mean={post_A.mean():.3f})')
axes[0].plot(x, post_B.pdf(x), 'r-', lw=2, label=f'Posterior B (mean={post_B.mean():.3f})')
axes[0].axvline(true_rate_A, color='b', ls='--', alpha=0.5, label='True A')
axes[0].axvline(true_rate_B, color='r', ls='--', alpha=0.5, label='True B')
axes[0].set_xlabel('Conversion rate'); axes[0].set_ylabel('Density')
axes[0].set_title('Beta Posteriors for A and B', fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].hist(lift_samples, bins=100, density=True, color='purple', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', lw=2, label='No difference')
axes[1].axvline(lift_samples.mean(), color='green', lw=2, ls='--', label=f'Mean lift={lift_samples.mean()*100:.2f}%')
axes[1].fill_betweenx([0, 40], np.percentile(lift_samples,2.5), np.percentile(lift_samples,97.5),
                      alpha=0.15, color='green', label='95% credible interval')
axes[1].set_xlabel('Lift (rate_B - rate_A)')
axes[1].set_title(f'P(B > A | data) = {P_B_beats_A:.3f}', fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('stats_01_ab_test.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_01_ab_test.png')

## Real-World Example 3: Base Rate Neglect and Comparison Summary

The most dangerous probability mistake: ignoring the prior (prevalence).
A 99% accurate test for a 0.1%-prevalent disease gives only ~9% PPV.
The same math governs fraud detection and any imbalanced classification task.

In [ ]:
# ============================================================
# Part A: Medical test -- base rate neglect
# ============================================================
def ppv_from_bayes(prevalence, sensitivity, specificity):
    # P(disease | positive) = P(pos|disease)*P(disease) / P(positive)
    P_D  = prevalence
    P_pos_given_D  = sensitivity
    P_pos_given_nD = 1.0 - specificity
    P_pos = P_pos_given_D * P_D + P_pos_given_nD * (1 - P_D)
    ppv = (P_pos_given_D * P_D) / P_pos
    return ppv

prev = 0.001   # 0.1% prevalence
sens = 0.99    # 99% sensitivity
spec = 0.99    # 99% specificity
ppv  = ppv_from_bayes(prev, sens, spec)

print('Medical Test Example')
print(f'  Prevalence:   {prev*100:.1f}%')
print(f'  Sensitivity:  {sens*100:.0f}%')
print(f'  Specificity:  {spec*100:.0f}%')
print(f'  PPV (P(disease | positive)): {ppv*100:.2f}%  -- much less than 99%!')

# Intuition: breakdown over 100,000 people
n = 100_000
TP = round(n * prev * sens)
FP = round(n * (1 - prev) * (1 - spec))
print(f'  In {n:,} tested: {TP} true positives, {FP} false positives')
print(f'  P(disease | positive) = {TP}/{TP+FP} = {TP/(TP+FP):.4f}')

# Effect of prevalence on PPV
print('\nPrevalence vs PPV (99% accurate test):')
prevalences = [0.001, 0.01, 0.05, 0.10, 0.20, 0.50]
ppvs = [ppv_from_bayes(p, 0.99, 0.99) for p in prevalences]
for p, v in zip(prevalences, ppvs):
    bar = '#' * int(v * 40)
    print(f'  Prevalence {p*100:5.1f}%  PPV={v*100:5.1f}%  {bar}')

# ============================================================
# Part B: Fraud detection
# ============================================================
fraud_rate = 0.001
ml_sens    = 0.95
ml_spec    = 0.990
ppv_fraud  = ppv_from_bayes(fraud_rate, ml_sens, ml_spec)

print('\nFraud Detection Example')
print(f'  Fraud rate:  {fraud_rate*100:.1f}%')
print(f'  Model sens:  {ml_sens*100:.0f}%,  spec: {ml_spec*100:.1f}%')
print(f'  P(fraud | flagged): {ppv_fraud*100:.2f}%')
print(f'  => auto-blocking all flagged would block {(1-ppv_fraud)*100:.0f}% legitimate transactions')

# ============================================================
# Part C: Frequentist vs Bayesian comparison table
# ============================================================
print('\nFrequentist vs Bayesian Summary')
rows = [
    ('Probability of', 'long-run frequency', 'degree of belief'),
    ('Parameters', 'fixed unknowns', 'random variables'),
    ('Key output', 'p-value', 'P(hypothesis | data)'),
    ('Uncertainty', 'confidence interval', 'credible interval'),
    ('Prior info', 'ignored', 'encoded as prior'),
    ('Small samples', 'CLT may not hold', 'prior regularizes'),
]
print(f'  {'Aspect':<20} {'Frequentist':<25} Bayesian')
print('  ' + '-' * 60)
for r in rows:
    print(f'  {r[0]:<20} {r[1]:<25} {r[2]}')

# ============================================================
# Part D: Plot prevalence vs PPV
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4))
prev_range = np.logspace(-3, -0.3, 200)
ppv_range  = [ppv_from_bayes(p, 0.99, 0.99) for p in prev_range]
ax.semilogx(prev_range * 100, [v * 100 for v in ppv_range], 'b-', lw=2)
ax.scatter([p*100 for p in prevalences], [v*100 for v in ppvs], color='red', zorder=5, s=60)
ax.axhline(50, color='gray', ls='--', label='50% threshold')
ax.set_xlabel('Disease Prevalence (%)')
ax.set_ylabel('PPV = P(disease | positive) %')
ax.set_title('Base Rate Neglect: 99%-Accurate Test Can Still Be Misleading', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stats_01_base_rate.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_01_base_rate.png')

# ============================================================
# Key Takeaways
# ============================================================
print('\nKEY TAKEAWAYS')
print('1. P(A|B) != P(B|A)  -- the direction of conditioning matters')
print('2. Conditional prob = joint / marginal -- always normalize')
print('3. Independence: P(A,B)=P(A)*P(B) -- test with KL divergence')
print('4. Low base rates dominate PPV even for highly accurate models')
print('5. Use log-probabilities to avoid underflow in multiplicative pipelines')
print('6. Bayesian P(B>A) is more actionable than frequentist p-value')
print('7. Naive Bayes assumes conditional independence -- verify with MI')

print('\nEXERCISES:')
print('1. Change prevalence to 5%: compute new PPV for the medical test')
print('2. Build an independent joint distribution and verify KL = 0')
print('3. Add a third variant C to the A/B test, compute P(C best)')